<a href="https://colab.research.google.com/github/ferjoseco/Expense-Tracker/blob/main/Final_Version_%20Expense_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# The first step is to import all the tools we need
import sqlite3  # This helps us work with databases (like a digital notebook)
import matplotlib.pyplot as plt  # This helps us make graphs
import seaborn as sns  # This makes our graphs colorful and better looking
import numpy as np  # This helps with math equations

# Opening our digital notebook (database) - if it doesn't exist, we'll make a new one
conn = sqlite3.connect("expenses.db")
# Cursor works like a pencil that lets us write in our notebook
cursor = conn.cursor()

# Creating a page in our notebook called "expenses" if it doesn't exist already
# This page has 4 columns: ID ("name" of the expense), amount (how much money was it),
# category (what type of expense), and date (when it happened)
# They are classified as INTEGER for numerical values and as TEXT for string values.
cursor.execute("""
CREATE TABLE IF NOT EXISTS expenses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    amount INTEGER NOT NULL,
    category TEXT NOT NULL,
    date TEXT NOT NULL
)
""")
# Saving our new page
conn.commit()

# This is how we add a new expense to our notebook
def add_expense():
    # Ask how much money was spent
    # Input is used for the user to be able to give us a value.
    # Int is used for the user to only input numerical values for error handling
    expense_amount = int(input("Enter the amount of your expense: "))

    # If it's a lot of money (more than 100), ask what type of expense it is
    if expense_amount > 100:
        category = input("How would you categorize this expense? (overhead, salaries, marketing): ")
    else:
        # If it's not much money (lower than 100), just call it "low expense"
        print("Low expense")
        category = "low expense"

    # Check if the category is one we know about (from the three options given above)
    # if function is used for error handling
    if category in ["overhead", "salaries", "marketing", "low expense"]:
        # Ask the date of the expense with another input function and define the variable "date"
        date = input("Enter the date of the expense (YYYY-MM-DD): ")
        # Write it down in our notebook with the cursor
        cursor.execute("INSERT INTO expenses (amount, category, date) VALUES (?, ?, ?)", (expense_amount, category, date))
        # Save what we wrote
        conn.commit()
        print("Expense added successfully!")
    else:
        # If the category is not one of the three given above, don't write it down and print an error
        print("Invalid category. Expense not added.")

# FUNCTION: EDIT AN EXPENSE
# So we can fix any mistakes the user has made
def edit_expense():
    # First, show all expenses so the user can pick which one to change
    view_expenses()
    try:
        # Ask "Which expense do you want to change? (give me its ID number)"
        # Int(Input) is used again for the user to give a value
        expense_id = int(input("Enter the ID of the expense you want to edit: "))
        # Find that expense in our notebook
        cursor.execute("SELECT * FROM expenses WHERE id = ?", (expense_id,))
        expense = cursor.fetchone()

        # "If" function is used to find the expense
        if expense:
            print("Editing the following expense:", expense)

            # Ask for the new information for the edit
            # Define expense amount, category and date variables again and ask for new inputs
            expense_amount = int(input("Enter the new amount of your expense: "))
            category = input("Enter the new category (overhead, salaries, marketing, low expense): ")
            date = input("Enter the new date of the expense (YYYY-MM-DD): ")

            # Erase the old info and write the new info
            cursor.execute("UPDATE expenses SET amount = ?, category = ?, date = ? WHERE id = ?",
                         (expense_amount, category, date, expense_id))
            # Save the changes
            conn.commit()
            print("Expense updated successfully!")
            # Error handling for a non-existent ID
        else:
            print("Expense ID not found.")
    except ValueError:
        # If someone types something that's not a number (more error handling)
        print("Invalid input. Please enter a valid number.")

# FUNCTION: DELETE AN EXPENSE
# So we can remove expenses we don't want anymore
def delete_expense():
    # First, show all expenses so we know which one to erase
    view_expenses()
    try:
        # Ask "Which expense do you want to erase? (give me its ID number)"
        # Int(input) is used again for values only
        expense_id = int(input("Enter the ID of the expense you want to delete: "))
        # Find that expense in our notebook
        cursor.execute("SELECT * FROM expenses WHERE id = ?", (expense_id,))
        expense = cursor.fetchone()

        # If function to see if the expense is found
        if expense:
            print("Deleting the following expense:", expense)
            # Erase it from our notebook using the following function
            cursor.execute("DELETE FROM expenses WHERE id = ?", (expense_id,))
            # Save our changes
            conn.commit()
            # Confirm the elimination of the expense to the user by printing this phrase
            print("Expense deleted successfully!")
        # Error handling using Else function
        else:
            print("Expense ID not found.")
    except ValueError:
        # Error handling if someone types something that's not a number
        print("Invalid input. Please enter a valid number.")


# FUNCTION: VIEW ALL EXPENSES
# So we can see everything the company money on, detailed with category and dates
def view_expenses():
    # Read all the expenses from our notebook
    cursor.execute("SELECT * FROM expenses")
    #fetchall is used to see all the expenses rather than only the first one
    expenses = cursor.fetchall()

    # Error handling if there are no expenses yet
    if not expenses:
        print("No expenses recorded yet.")
    #printing all the data gathered
    else:
        print("\nHere are your recorded expenses:")
        # Show each expense one by one using a loop
        for expense in expenses:
            print(f"ID: {expense[0]}, Amount: {expense[1]}, Category: {expense[2]}, Date: {expense[3]}")

# FUNCTION: SHOW EXPENSES AS BAR GRAPH
#So we can visualize the data inputed by the user
def visualize_expenses_bar():
    # Get all expenses and add up amounts for each category
    cursor.execute("SELECT category, SUM(amount) FROM expenses GROUP by category")
    #use fetchall to gather all the expenses that have been inputted
    expenses = cursor.fetchall()

    # Get the category names (defined before as "overhead", "marketing", salaries and "low expenses")
    categories = [x[0] for x in expenses]
    # Get the total amounts for each category
    amounts = [x[1] for x in expenses]

    # Draw the bar graph with "shrek green" bars
    # Use the HEXCODE #B0C400 for shreek green (get out of my swamp)
    plt.bar(categories, amounts, color="#B0C400")
    # Add titles to the graph with plt funtions
    plt.title("Expenses by Category")
    plt.xlabel("Category")
    plt.ylabel("Amount")
    # Show the graph (but don't wait for it to close using block=false)
    plt.show(block=False)

# FUNCTION: SHOW EXPENSES AS PIE CHART
#So we can visualize the data inputed by the user in a pie chart
def visualize_expenses_pie():
    # Get all expenses and add up amounts for each category
    cursor.execute("SELECT category, SUM(amount) FROM expenses GROUP by category")
    expenses = cursor.fetchall()

    # Get the category names (defined before as "overhead", "marketing", salaries and "low expenses")
    categories = [x[0] for x in expenses]
    # Get the total amounts for each category
    amounts = [x[1] for x in expenses]

    # Draw the pie chart with colorful slices (HSV palette)
    plt.pie(amounts, labels=categories, autopct='%1.1f%%', colors=sns.color_palette('hsv'))
    # Add a title to the graph usinf plt functions
    plt.title("Expenses by Category")
    # Show the graph (but don't wait for it to close using block=false)
    plt.show(block=False)

# MAIN MENU FUNCTION
# This is the remote control for our program for the user to decide the use of the model
def main():
    while True:
        # Show the menu options with a print function
        print("\nMenu:")
        print("1. Add Expense")
        print("2. Edit Expense")
        print("3. Delete Expense")
        print("4. View Expenses")
        print("5. Visualise Expenses as Bar Chart")
        print("6. Visualise Expenses as Pie Chart")
        print("7. Exit")

        # Ask the user "What do you want to do?" and give them the options
        choice = input("Enter your choice: ")

        # Do different things based on the choice usinf the if, elif and else function
        # Call any of the variables defined at the top of the code
        if choice == "1":
            add_expense()
        elif choice == "2":
            edit_expense()
        elif choice == "3":
            delete_expense()
        elif choice == "4":
            view_expenses()
        elif choice == "5":
            visualize_expenses_bar()
        elif choice == "6":
            visualize_expenses_pie()
        elif choice == "7":
            print("Exiting the program. Goodbye!")
            break  # Turn off the program
        # Error handling for menu input mistakes
        else:
            print("Invalid choice. Please try again.")

# START THE PROGRAM
# Start the (main menu)
main()
# Close our digital notebook when the user is done
conn.close()


Menu:
1. Add Expense
2. Edit Expense
3. Delete Expense
4. View Expenses
5. Visualise Expenses as Bar Chart
6. Visualise Expenses as Pie Chart
7. Exit


In [ ]:
# Begin by importing necessary libraries
# pandas for data manipulation
# plt for data visualisation
import pandas as pd
import matplotlib.pyplot as plt

# load data from excel file
# print to load first few rows
data = pd.read_excel("ASSESSMENT1DATA.xlsx")
print(data.head())

# import regression tools
# train_test_split to divide into training and test sets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# define variables; independent as expenses
# dependent as profit
X = data[['EXPENSE']]
y = data['PROFIT']

# split test set (0.2 or 20%) and training (.8 or 80%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# intercept = profit when expense = 0
# coefficient = amount profit changes per unit change in expense
# print model coefficients
print(f'Intercept: {model.intercept_}, Coefficient: {model.coef_}')

# predict profit vals
y_pred = model.predict(X_test)
from sklearn.metrics import mean_squared_error, r2_score

# calculate metrics
# mean sq. error is avg ^2 diff between actual and predicted vals
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
# r-squared is proportion of variance in profit, 0-1, higher = better fit
print(f'R-squared: {r2_score(y_test, y_pred)}')

# visualise results
# scatterplot as shown by brown dots
plt.scatter(X_test, y_test, color='#795A2D', label='Actual')
# regression line as shown by shrek green line
plt.plot(X_test, y_pred, color='#B0C400', label='Predicted')
# x and y axis labels
plt.xlabel('Expense')
plt.ylabel('Profit')
# legend
plt.legend()
# final command to display scatterplot
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'ASSESSMENT1DATA.xlsx'